In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install transformers torchaudio librosa pandas numpy scikit-learn tqdm


In [ ]:
import os
import pandas as pd

TAMIL_PATH = "/content/drive/MyDrive/DD_DL_Depression/Tamil"

rows = []

for folder in ["depressed", "nondepressed"]:
    label = "Depressed" if folder == "depressed" else "Non-Depressed"
    folder_path = os.path.join(TAMIL_PATH, folder)

    for file in os.listdir(folder_path):
        if file.endswith(".wav"):
            rows.append([os.path.join(folder_path, file), label])

tamil_df = pd.DataFrame(rows, columns=["audio_path", "label"])
tamil_df.head()


,audio_path,label
0,/content/drive/MyDrive/DD_DL_Depression/Tamil/...,Depressed
1,/content/drive/MyDrive/DD_DL_Depression/Tamil/...,Depressed
2,/content/drive/MyDrive/DD_DL_Depression/Tamil/...,Depressed
3,/content/drive/MyDrive/DD_DL_Depression/Tamil/...,Depressed
4,/content/drive/MyDrive/DD_DL_Depression/Tamil/...,Depressed


In [ ]:
MAL_PATH = "/content/drive/MyDrive/DD_DL_Depression/Malayalam"

rows = []

for folder in ["depressed", "nondepressed"]:
    label = "Depressed" if folder == "depressed" else "Non-Depressed"
    folder_path = os.path.join(MAL_PATH, folder)

    for file in os.listdir(folder_path):
        if file.endswith(".wav"):
            rows.append([os.path.join(folder_path, file), label])

mal_df = pd.DataFrame(rows, columns=["audio_path", "label"])
mal_df.head()


,audio_path,label
0,/content/drive/MyDrive/DD_DL_Depression/Malaya...,Depressed
1,/content/drive/MyDrive/DD_DL_Depression/Malaya...,Depressed
2,/content/drive/MyDrive/DD_DL_Depression/Malaya...,Depressed
3,/content/drive/MyDrive/DD_DL_Depression/Malaya...,Depressed
4,/content/drive/MyDrive/DD_DL_Depression/Malaya...,Depressed


In [ ]:
import torch
import torchaudio
from transformers import Wav2Vec2Processor, Wav2Vec2Model

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base").to(device)
model.eval()


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Wav2Vec2Model(
  (feature_extractor): Wav2Vec2FeatureEncoder(
    (conv_layers): ModuleList(
      (0): Wav2Vec2GroupNormConvLayer(
        (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
        (activation): GELUActivation()
        (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
      )
      (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
      (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
    )
  )
  (feature_projection): Wav2Vec2FeatureProjection(
    (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (projection): Linear(in_features=512, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): Wav2Vec2Encoder(
    (pos_conv_embed): Wav2Vec2PositionalConvEmbedding(
  

In [ ]:
def get_features(path):
    speech, sr = torchaudio.load(path)
    speech = speech.mean(dim=0)

    if sr != 16000:
        resample = torchaudio.transforms.Resample(sr, 16000)
        speech = resample(speech)

    inputs = processor(speech, sampling_rate=16000, return_tensors="pt", padding=True)

    with torch.no_grad():
        outputs = model(inputs.input_values.to(device))

    return outputs.last_hidden_state.mean(dim=1).cpu().numpy()[0]


In [ ]:
import numpy as np
from tqdm import tqdm

X_t, y_t = [], []

for _, row in tqdm(tamil_df.iterrows(), total=len(tamil_df)):
    X_t.append(get_features(row["audio_path"]))
    y_t.append(row["label"])

X_t = np.array(X_t)
y_t = np.array(y_t)


100%|██████████| 135/135 [00:06<00:00, 20.42it/s]


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X_tr, X_val, y_tr, y_val = train_test_split(
    X_t, y_t, test_size=0.2, random_state=42
)

tamil_model = LogisticRegression(max_iter=1000)
tamil_model.fit(X_tr, y_tr)

pred = tamil_model.predict(X_val)
print(classification_report(y_val, pred))


               precision    recall  f1-score   support

    Depressed       0.94      1.00      0.97        16
Non-Depressed       1.00      0.91      0.95        11

     accuracy                           0.96        27
    macro avg       0.97      0.95      0.96        27
 weighted avg       0.97      0.96      0.96        27



In [ ]:
import numpy as np
from tqdm import tqdm

X_t, y_t = [], []

for _, row in tqdm(mal_df.iterrows(), total=len(mal_df)):
    X_t.append(get_features(row["audio_path"]))
    y_t.append(row["label"])

X_t = np.array(X_t)
y_t = np.array(y_t)


100%|██████████| 121/121 [00:11<00:00, 10.76it/s]


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X_tr, X_val, y_tr, y_val = train_test_split(
    X_t, y_t, test_size=0.2, random_state=42
)

mal_model = LogisticRegression(max_iter=1000)
mal_model.fit(X_tr, y_tr)

pred = mal_model.predict(X_val)
print(classification_report(y_val, pred))


               precision    recall  f1-score   support

    Depressed       1.00      1.00      1.00        14
Non-Depressed       1.00      1.00      1.00        11

     accuracy                           1.00        25
    macro avg       1.00      1.00      1.00        25
 weighted avg       1.00      1.00      1.00        25



In [ ]:
tamil_test_path = "/content/drive/MyDrive/DD_DL_Depression/Tamil/test"

tamil_files = sorted(os.listdir(tamil_test_path))
tamil_preds = []

for file in tqdm(tamil_files):
    feat = get_features(os.path.join(tamil_test_path, file))
    tamil_preds.append(tamil_model.predict([feat])[0])


100%|██████████| 101/101 [00:09<00:00, 10.19it/s]


In [ ]:
mal_test_path = "/content/drive/MyDrive/DD_DL_Depression/Malayalam/test" # Corrected 'malayalam' to 'Malayalam'

mal_files = sorted(os.listdir(mal_test_path))
mal_preds = []

for file in tqdm(mal_files):
    feat = get_features(os.path.join(mal_test_path, file))
    mal_preds.append(mal_model.predict([feat])[0])

100%|██████████| 74/74 [00:03<00:00, 23.55it/s]


In [ ]:
import os
os.makedirs("/content/drive/MyDrive/DD_DL_Depression/outputs", exist_ok=True)


In [ ]:
tamil_out = pd.DataFrame({
    "file_name": tamil_files,
    "label": tamil_preds
})

tamil_out.to_csv(
    "/content/drive/MyDrive/DD_DL_Depression/outputs/codex_Tamil.csv",
    index=False
)


In [ ]:
mal_out = pd.DataFrame({
    "file_name": mal_files,
    "label": mal_preds
})

mal_out.to_csv(
    "/content/drive/MyDrive/DD_DL_Depression/outputs/codex_Malayalam.csv",
    index=False
)


In [ ]:
%cd /content/drive/MyDrive/DD_DL_Depression/outputs
!zip codex_depression.zip codex_Tamil.csv codex_Malayalam.csv


/content/drive/MyDrive/DD_DL_Depression/outputs
  adding: codex_Tamil.csv (deflated 87%)
  adding: codex_Malayalam.csv (deflated 86%)
